# 03. B078 250m OD 셀 공간 매핑

B078의 출발·도착 250m 셀 좌표를 행정동 경계에 결합하고, 동대문구 출발 목적지 마트를 만듭니다. 공개 표본은 코드·좌표계·처리 흐름 검증용입니다. 전체 원본과 동일한 분석 결과로 해석하지 않습니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
PROJECT_DIR = '/content/drive/MyDrive/막시무스'  # 실제 경로로 수정
%cd $PROJECT_DIR
!pip -q install -r requirements-eda.txt

In [ ]:
from pathlib import Path
import pandas as pd

B078_PATH = Path('data/external/B078_PURPOSE_250M_202403_sample.csv')
BOUNDARY_PATH = Path('data/external/seoul_administrative_dongs_20260701.geojson')
if not B078_PATH.exists() or not BOUNDARY_PATH.exists():
    raise FileNotFoundError('B078 파일과 행정동 경계 파일을 준비하세요.')

sample = pd.read_csv(B078_PATH, encoding='cp949', nrows=5)
display(sample)
print(f'열 수: {sample.shape[1]}')

## 1. 행정동 공간 결합

B078 셀의 EPSG:5179 좌표를 WGS84로 바꾸고, 점이 포함되는 행정동 폴리곤을 찾습니다. 최종 본분석에서는 B078 관측 시점과 같은 기준연도의 경계·행정동 코드 연계표를 사용해야 합니다.

In [ ]:
import subprocess, sys
subprocess.run([
    sys.executable, '-m', 'scripts.build_b078_dong_mart',
    '--input', str(B078_PATH),
    '--boundary', str(BOUNDARY_PATH),
    '--max-minutes', '30',
], check=True)

crosswalk = pd.read_csv('data/processed/b078_dong_mart/b078_cell_admin_dong_crosswalk.csv')
audit = Path('data/processed/b078_dong_mart/b078_mapping_audit.json').read_text(encoding='utf-8')
display(crosswalk.head())
print(audit)

## 2. 전체 원본을 받았을 때

1. `B078_PATH`만 승인된 전체 파일 경로로 교체합니다.
2. 코드북으로 목적 코드, 시간 단위, 좌표계, 마스킹 규칙을 확인합니다.
3. 행정동 경계는 관측 기간과 맞추고, 셀 매핑 성공률·경계 변경률을 기록합니다.
4. 산출물은 반출 정책이 허용하는 집계표·계수·성능지표만 공유합니다.